In [1]:
from IPython.display import display, HTML
display(HTML("""
<style>
div.container{width:90% !important;}
div.cell.code_cell.rendered{width:100%;}
div.input_prompt{padding:0px;}
div.CodeMirror {font-family:Consolas; font-size:20pt;}
.inner_cell{font-size:20pt;}
div.text_cell_render pre code {font-size:20pt; line-height:30px;}
div.output {font-size:20pt; font-weight:bold;}
div.input {font-family:Consolas; font-size:20pt;}
div.prompt {min-width:70px;}
div#toc-wrapper{padding-top:120px;}
div.text_cell_render ul li{font-size:20pt;padding:5px;}
table.dataframe{font-size:20px;}
</style>
"""))

<font color="red" size="6"><b>ch14. 웹 데이터 수집</b></font>

# 1절. BeautifulSoup과 parser
    (정적 웹크롤링, 공공api사용)
    
`pip install bs4` 아나콘다를 설치하면 자동 설치되는 패키지에 포함
- 공식 사이트 : https://www.crummy.com/software/BeautifulSoup/
- documentation : https://www.crummy.com/software/BeautifulSoup/bs4/doc/

In [2]:
import requests # HTTP요청 처리하는 lib
# file:// == c:
# http://www.
from requests_file import FileAdapter

In [12]:
# 로컬에 있는 파일을 웹요청하듯이 읽어오는 작업
s = requests.Session()
s.mount("file://", FileAdapter()) # file://로 시작하는 url을 어댑터가 처리
response = s.get("file:///ai/lecNote/01_python/data/ch14_sample.html") # c:/ai/lecNote/01_python/data/ch14_sample.html
response

<Response [200]>

In [13]:
if response:
    print('해당 url에 접근함')
else:
    print('해당 url에 거부됨')

해당 url에 접근함


In [14]:
response.status_code
# 200 : 정상
# 404 : 없는 페이지

200

In [15]:
response.content # html의 바이너리 형식의 내용

b'<!DOCTYPE html>\r\n<html lang="en">\r\n<head>\r\n  <meta charset="UTF-8">\r\n</head>\r\n<body>\r\n  <h1 class="greeting css" id="text">Hello, CSS</h1>\r\n  <h1 class="css">Hi, CSS</h1>\r\n  <div id="subject">subject \xec\x84\xa0\xed\x83\x9d\xec\x9e\x90 \xec\x95\x88\xec\x9d\x98 \xeb\x82\xb4\xec\x9a\xa9</div>\r\n  <p>CSS \xec\x84\xa0\xed\x83\x9d\xec\x9e\x90\xeb\x8a\x94 \xeb\x8b\xa4\xec\x96\x91\xed\x95\x9c \xea\xb3\xb3\xec\x97\x90\xec\x84\x9c \xed\x99\x9c\xec\x9a\xa9\xeb\x90\xa9\xeb\x8b\x88\xeb\x8b\xa4</p>\r\n  <div class="contents">\r\n    \xec\x84\xa0\xed\x83\x9d\xec\x9e\x90\xeb\xa5\xbc \xec\x96\xb4\xeb\x96\xbb\xea\xb2\x8c \xec\x9e\x91\xec\x84\xb1\xed\x95\x98\xeb\x8a\x90\xeb\x83\x90\xec\x97\x90 \xeb\x94\xb0\xeb\x9d\xbc\r\n    <span>\xeb\x8b\xa4\xeb\xa5\xb8<b>\xec\x9a\x94\xec\x86\x8c\xea\xb0\x80 \xeb\xb0\x98\xed\x99\x98</b></span>\xeb\x90\xa9\xeb\x8b\x88\xeb\x8b\xa4\r\n  </div>\r\n  <div>CSS \xec\x84\xa0\xed\x83\x9d\xec\x9e\x90\xeb\x8a\x94 \xeb\x8b\xa4\xec\x96\x91\xed\x95\x9c \xea\xb3\

In [16]:
print(response.content.decode('utf-8'))

<!DOCTYPE html>
<html lang="en">
<head>
  <meta charset="UTF-8">
</head>
<body>
  <h1 class="greeting css" id="text">Hello, CSS</h1>
  <h1 class="css">Hi, CSS</h1>
  <div id="subject">subject 선택자 안의 내용</div>
  <p>CSS 선택자는 다양한 곳에서 활용됩니다</p>
  <div class="contents">
    선택자를 어떻게 작성하느냐에 따라
    <span>다른<b>요소가 반환</b></span>됩니다
  </div>
  <div>CSS 선택자는 다양한 곳에 <b>활용</b>됩니다</div>
</body>
</html>


In [19]:
response.text

'<!DOCTYPE html>\r\n<html lang="en">\r\n<head>\r\n  <meta charset="UTF-8">\r\n</head>\r\n<body>\r\n  <h1 class="greeting css" id="text">Hello, CSS</h1>\r\n  <h1 class="css">Hi, CSS</h1>\r\n  <div id="subject">subject 선택자 안의 내용</div>\r\n  <p>CSS 선택자는 다양한 곳에서 활용됩니다</p>\r\n  <div class="contents">\r\n    선택자를 어떻게 작성하느냐에 따라\r\n    <span>다른<b>요소가 반환</b></span>됩니다\r\n  </div>\r\n  <div>CSS 선택자는 다양한 곳에 <b>활용</b>됩니다</div>\r\n</body>\r\n</html>'

In [24]:
# html 파싱 객체
from bs4 import BeautifulSoup
soup = BeautifulSoup(response.text, #response.content, 
                    "html.parser")
# soup

In [37]:
# 1. soup.select_one('선택자') : 해당 선택자 처음 하나 엘리먼트만 
el = soup.select_one('h1.css')
print('el =>', el)
print('el.text   =>', el.text)
print('el.string =>', el.string)
print('el의 속성들 =>', el.attrs)
print('el의 class속성 =>', el.attrs['class'])
print('el의 class속성 =>', el.attrs.get('class'))
# print('el의 href속성(없는 속성은 에러) =>', el.attrs.get['href'])
print('el의 href속성 =>', el.attrs.get('href'))
print('el의 name =>', el.name)

el => <h1 class="greeting css" id="text">Hello, CSS</h1>
el.text   => Hello, CSS
el.string => Hello, CSS
el의 속성들 => {'class': ['greeting', 'css'], 'id': 'text'}
el의 class속성 => ['greeting', 'css']
el의 class속성 => ['greeting', 'css']
el의 href속성 => None
el의 name => h1


In [45]:
# 2. soup.select('선택자') : 해당 선택자 엘리먼트 다 list로
els = soup.select('h1.css')
print('els =>', els)
print('els들의 text =>', [el.text for el in els])
print('els들의 string =>', [el.string for el in els])
print('els들의 속성들 =>', [el.attrs for el in els])
print('els들의 class 속성 =>', [el.attrs.get('class') for el in els])

els => [<h1 class="greeting css" id="text">Hello, CSS</h1>, <h1 class="css">Hi, CSS</h1>]
els들의 text => ['Hello, CSS', 'Hi, CSS']
els들의 string => ['Hello, CSS', 'Hi, CSS']
els들의 속성들 => [{'class': ['greeting', 'css'], 'id': 'text'}, {'class': ['css']}]
els들의 class 속성 => [['greeting', 'css'], ['css']]


In [50]:
# 3. soup.find(태그, 속성)  vs soup.select_one('선택자') : 해당 속성을 갖은 태그 처음 하나만 
print('select_one :', soup.select_one('h1.css'))
print('find       :', soup.find('h1', {'class':'css'}))
print('find       :', soup.find('h1', class_='css'))
print()
print('select_one :', soup.select_one('h1#text'))
print('select_one :', soup.find('h1', {'id':'text'}))

select_one : <h1 class="greeting css" id="text">Hello, CSS</h1>
find       : <h1 class="greeting css" id="text">Hello, CSS</h1>
find       : <h1 class="greeting css" id="text">Hello, CSS</h1>

select_one : <h1 class="greeting css" id="text">Hello, CSS</h1>
select_one : <h1 class="greeting css" id="text">Hello, CSS</h1>


In [61]:
# 4. soup.find_all(태그, 속성) vs. soup.select('선택자') : 해당 엘리먼트 다 list로
print('모든 h1.css와 span태그 :', soup.select('h1.css, span'))
print('모든 h1.css와 span태그 :', soup.find_all(['h1'], class_='css') +
                                soup.find_all('span'))

모든 h1.css와 span태그 : [<h1 class="greeting css" id="text">Hello, CSS</h1>, <h1 class="css">Hi, CSS</h1>, <span>다른<b>요소가 반환</b></span>]
모든 h1.css와 span태그 : [<h1 class="greeting css" id="text">Hello, CSS</h1>, <h1 class="css">Hi, CSS</h1>, <span>다른<b>요소가 반환</b></span>]


In [70]:
# 없는 엘리먼트 찾기
print('find_all(빈list) :', soup.find_all('a'))
print('find(None)       :', soup.find('a').text)
print('select(빈list) :', soup.select('a'))
print('select_one(None) :', soup.select_one('a'))

find_all(빈list) : []


AttributeError: 'NoneType' object has no attribute 'text'